# Curve fitting

Curve fitting means adjusting a model's parameters so that its predictions are close to the observed data. Neural networks use the same basic idea, but with more flexible models and many more parameters.

In this notebook, you will manually tune three simple models:

1. a line with one input feature;
2. a plane with two input features; and
3. a logistic regression model for binary classification.

> **Using this notebook in Google Colab:** select **Runtime → Run all**, then move the sliders. You do not need to edit the code. If a widget does not appear, run its cell again with the play button on the left.

The data are synthetic, so they are intended for learning rather than clinical interpretation.

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Layout, interact
from IPython.display import HTML, display

SLIDER_STYLE = {'description_width': 'initial'}
SLIDER_LAYOUT = Layout(width='550px')


def mean_squared_error(observed, predicted):
    """Average squared distance between observations and predictions."""
    return np.mean((observed - predicted) ** 2)


def binary_cross_entropy(observed, probability):
    """Loss for binary predictions; lower values indicate a better fit."""
    probability = np.clip(probability, 1e-10, 1 - 1e-10)
    return -np.mean(
        observed * np.log(probability)
        + (1 - observed) * np.log(1 - probability)
    )


## Activity 1: Fit a line

The model is `prediction = slope × x + intercept`. Move the sliders to place the red line through the data. Your goal is to make the **mean squared error (MSE)** as small as possible.

In [2]:
# Create reproducible data that roughly follow a straight line.
rng = np.random.default_rng(42)
line_x = rng.uniform(-10, 10, size=500)
line_y = 2 * line_x + 1 + rng.normal(0, 2, size=line_x.size)

# Embed the data in a browser-side widget. Slider changes do not call Python.
line_widget = """
<div id="line-fitting-widget" class="curve-widget">
  <style>
    #line-fitting-widget {
      max-width: 900px; padding: 16px; border: 1px solid #d9d9d9;
      border-radius: 10px; font-family: Arial, sans-serif; background: white;
    }
    #line-fitting-widget .controls { display: grid; gap: 10px; margin-bottom: 8px; }
    #line-fitting-widget label { display: grid; grid-template-columns: 90px 1fr 60px; gap: 10px; align-items: center; }
    #line-fitting-widget input[type=range] { width: 100%; }
    #line-fitting-widget output { font-variant-numeric: tabular-nums; text-align: right; }
    #line-fitting-widget .metric { margin: 10px 0 4px; font-size: 17px; font-weight: 600; }
    #line-fitting-widget svg { width: 100%; height: auto; display: block; }
    #line-fitting-widget .axis-label { fill: #333; font-size: 15px; }
    #line-fitting-widget .tick-label { fill: #555; font-size: 12px; }
  </style>
  <div class="controls">
    <label>Slope
      <input data-role="slope" type="range" min="-2.5" max="2.5" step="0.05" value="-2">
      <output data-role="slope-value">-2.00</output>
    </label>
    <label>Intercept
      <input data-role="intercept" type="range" min="-10" max="10" step="0.1" value="-10">
      <output data-role="intercept-value">-10.0</output>
    </label>
  </div>
  <div class="metric">MSE: <span data-role="mse"></span> <small>(lower is better)</small></div>
  <svg data-role="plot" viewBox="0 0 900 500" role="img" aria-label="Observed data and fitted line"></svg>
</div>
<script>
(() => {
  const root = document.getElementById('line-fitting-widget');
  const svg = root.querySelector('[data-role=plot]');
  const x = __LINE_X__;
  const y = __LINE_Y__;
  const ns = 'http://www.w3.org/2000/svg';
  const area = {left: 65, right: 875, top: 20, bottom: 445};
  const xMin = Math.min(...x);
  const xMax = Math.max(...x);
  const yMin = Math.min(...y, -36);
  const yMax = Math.max(...y, 36);
  const scaleX = value => area.left + (value - xMin) / (xMax - xMin) * (area.right - area.left);
  const scaleY = value => area.bottom - (value - yMin) / (yMax - yMin) * (area.bottom - area.top);

  function addSvg(tag, attributes, parent = svg) {
    const element = document.createElementNS(ns, tag);
    for (const [name, value] of Object.entries(attributes)) element.setAttribute(name, value);
    parent.appendChild(element);
    return element;
  }

  // Draw axes, grid lines, and labels once.
  for (let i = 0; i <= 5; i++) {
    const xValue = xMin + i * (xMax - xMin) / 5;
    const yValue = yMin + i * (yMax - yMin) / 5;
    const px = scaleX(xValue);
    const py = scaleY(yValue);
    addSvg('line', {x1: px, x2: px, y1: area.top, y2: area.bottom, stroke: '#e5e7eb'});
    addSvg('line', {x1: area.left, x2: area.right, y1: py, y2: py, stroke: '#e5e7eb'});
    const xTick = addSvg('text', {x: px, y: area.bottom + 20, 'text-anchor': 'middle', class: 'tick-label'});
    xTick.textContent = xValue.toFixed(0);
    const yTick = addSvg('text', {x: area.left - 10, y: py + 4, 'text-anchor': 'end', class: 'tick-label'});
    yTick.textContent = yValue.toFixed(0);
  }
  addSvg('line', {x1: area.left, x2: area.right, y1: area.bottom, y2: area.bottom, stroke: '#333'});
  addSvg('line', {x1: area.left, x2: area.left, y1: area.top, y2: area.bottom, stroke: '#333'});
  const xLabel = addSvg('text', {x: 470, y: 490, 'text-anchor': 'middle', class: 'axis-label'});
  xLabel.textContent = 'Input feature (x)';
  const yLabel = addSvg('text', {x: 16, y: 235, transform: 'rotate(-90 16 235)', 'text-anchor': 'middle', class: 'axis-label'});
  yLabel.textContent = 'Outcome (y)';

  const points = addSvg('g', {});
  for (let i = 0; i < x.length; i++) {
    addSvg('circle', {cx: scaleX(x[i]), cy: scaleY(y[i]), r: 2.4, fill: '#2563eb', opacity: 0.35}, points);
  }
  const modelLine = addSvg('line', {stroke: '#dc143c', 'stroke-width': 4});
  const slopeInput = root.querySelector('[data-role=slope]');
  const interceptInput = root.querySelector('[data-role=intercept]');

  function update() {
    const slope = Number(slopeInput.value);
    const intercept = Number(interceptInput.value);
    modelLine.setAttribute('x1', scaleX(xMin));
    modelLine.setAttribute('y1', scaleY(slope * xMin + intercept));
    modelLine.setAttribute('x2', scaleX(xMax));
    modelLine.setAttribute('y2', scaleY(slope * xMax + intercept));
    const mse = y.reduce((sum, observed, i) => {
      const error = observed - (slope * x[i] + intercept);
      return sum + error * error;
    }, 0) / y.length;
    root.querySelector('[data-role=slope-value]').textContent = slope.toFixed(2);
    root.querySelector('[data-role=intercept-value]').textContent = intercept.toFixed(1);
    root.querySelector('[data-role=mse]').textContent = mse.toFixed(2);
  }

  slopeInput.addEventListener('input', update);
  interceptInput.addEventListener('input', update);
  update();
})();
</script>
"""
line_widget = line_widget.replace('__LINE_X__', json.dumps(line_x.tolist()))
line_widget = line_widget.replace('__LINE_Y__', json.dumps(line_y.tolist()))
display(HTML(line_widget))


## Activity 2: Fit a plane using two input features

The model is `prediction = a × feature 1 + b × feature 2 + bias`. Adjust **a**, **b**, and **bias** until the surface fits the points. The orientation sliders only change your view; they do not change the model.

In [3]:
# Create a plotting grid and reproducible points from a plane.
plane_axis = np.linspace(-10, 10, 20)
plane_grid_x, plane_grid_y = np.meshgrid(plane_axis, plane_axis)

rng = np.random.default_rng(42)
plane_x = rng.uniform(-10, 10, size=100)
plane_y = rng.uniform(-10, 10, size=100)
plane_z = plane_x + plane_y


@interact(
    a=FloatSlider(value=0, min=-2, max=2, step=0.1, description='Coefficient a:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    b=FloatSlider(value=0, min=-2, max=2, step=0.1, description='Coefficient b:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    bias=FloatSlider(value=0, min=-5, max=5, step=0.1, description='Bias:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    azimuth=FloatSlider(value=45, min=0, max=360, step=5, description='Rotate:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    elevation=FloatSlider(value=30, min=0, max=90, step=5, description='Tilt:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
)
def plot_plane_fit(a, b, bias, azimuth, elevation):
    surface_z = a * plane_grid_x + b * plane_grid_y + bias
    predicted_z = a * plane_x + b * plane_y + bias
    mse = mean_squared_error(plane_z, predicted_z)

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(projection='3d')
    ax.plot_surface(
        plane_grid_x, plane_grid_y, surface_z,
        cmap='viridis', edgecolor='none', alpha=0.55,
    )
    ax.scatter(plane_x, plane_y, plane_z, color='crimson', s=22, label='Observed data')
    ax.set(
        title=f'Fit a plane — MSE: {mse:.2f} (lower is better)',
        xlabel='Feature 1',
        ylabel='Feature 2',
        zlabel='Outcome',
    )
    ax.view_init(elev=elevation, azim=azimuth)
    ax.legend()
    plt.show()
    plt.close(fig)


interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='Coefficient a:', layout=Lay…

## Activity 3: Logistic regression for binary classification

A binary classifier estimates the probability that each observation belongs to class 1. You can imagine the two features as two test results, although these data are entirely synthetic.

Adjust the two weights and the bias. Aim for a low **binary cross-entropy (BCE) loss** and high **accuracy**. The dashed line is the decision boundary: the model predicts class 1 on the side where the probability is above 50%.

In [4]:
def sigmoid(value):
    """Convert any real-valued score into a probability from 0 to 1."""
    value = np.clip(value, -500, 500)
    return 1 / (1 + np.exp(-value))


def logistic_probability(feature_0, feature_1, weight_0, weight_1, bias):
    score = weight_0 * feature_0 + weight_1 * feature_1 + bias
    return sigmoid(score)

In [5]:
# Create two reproducible groups with two features each.
rng = np.random.default_rng(42)
class_0 = rng.normal(loc=2, scale=1, size=(100, 2))
class_1 = rng.normal(loc=5, scale=1, size=(100, 2))
classification_features = np.vstack([class_0, class_1])
class_labels = np.concatenate([np.zeros(100), np.ones(100)])

# Create a grid on which to display the model's probabilities.
padding = 0.5
feature_0_axis = np.linspace(
    classification_features[:, 0].min() - padding,
    classification_features[:, 0].max() + padding,
    80,
)
feature_1_axis = np.linspace(
    classification_features[:, 1].min() - padding,
    classification_features[:, 1].max() + padding,
    80,
)
classification_grid_0, classification_grid_1 = np.meshgrid(
    feature_0_axis, feature_1_axis
)

In [6]:
@interact(
    weight_0=FloatSlider(value=1, min=-5, max=5, step=0.1, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Weight for feature 0:', continuous_update=False),
    weight_1=FloatSlider(value=-1, min=-5, max=5, step=0.1, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Weight for feature 1:', continuous_update=False),
    bias=FloatSlider(value=0, min=-20, max=20, step=0.1, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Bias:', continuous_update=False),
)
def plot_classifier(weight_0, weight_1, bias):
    grid_probability = logistic_probability(
        classification_grid_0, classification_grid_1, weight_0, weight_1, bias
    )
    observed_probability = logistic_probability(
        classification_features[:, 0], classification_features[:, 1],
        weight_0, weight_1, bias,
    )
    loss = binary_cross_entropy(class_labels, observed_probability)
    predicted_class = (observed_probability >= 0.5).astype(int)
    accuracy = np.mean(predicted_class == class_labels)

    fig, ax = plt.subplots(figsize=(9, 6))
    probability_map = ax.contourf(
        classification_grid_0, classification_grid_1, grid_probability,
        levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=0.35,
    )
    probability_varies = not np.allclose(grid_probability.min(), grid_probability.max())
    crosses_threshold = grid_probability.min() <= 0.5 <= grid_probability.max()
    if probability_varies and crosses_threshold:
        ax.contour(
            classification_grid_0, classification_grid_1, grid_probability,
            levels=[0.5], colors='black', linestyles='dashed', linewidths=2,
        )
    else:
        ax.text(
            0.02, 0.02, 'No 50% decision boundary is visible with these settings.',
            transform=ax.transAxes, fontsize=9,
            bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'gray'},
        )
    ax.scatter(class_0[:, 0], class_0[:, 1], color='royalblue', label='Class 0', alpha=0.75)
    ax.scatter(class_1[:, 0], class_1[:, 1], color='crimson', label='Class 1', alpha=0.75)
    colorbar = fig.colorbar(probability_map, ax=ax)
    colorbar.set_label('Predicted probability of class 1')
    ax.set(
        title=f'BCE loss: {loss:.3f} (lower is better) | Accuracy: {accuracy:.1%}',
        xlabel='Feature 0',
        ylabel='Feature 1',
    )
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()
    plt.close(fig)


interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='Weight for feature 0:', lay…

### Try this

Start by deciding whether each feature should increase or decrease the probability of class 1. Then adjust the bias to move the dashed boundary between the two groups.

## Take-home message

Each activity followed the same cycle: choose parameters, calculate predictions, measure the error, and adjust the parameters. Training a machine-learning model automates that adjustment step.